# 07 - Report: figures, Table 2, summary and submission material

Collects the results of notebooks 01–06 (from outputs/):
- **Figures** 1-3 (ROC curves, incremental odds ratios, confounding-free discrimination), PDF with embedded fonts.
- **Table 2** (main results) and outputs/results_summary.md (readings by the plan's rules, every result with where
  it goes in the paper, and alerts).
- **TRIPOD+AI** checklist pre-filled with where each item is answered.
- **Preprint package** (outputs/preprint_package/ and .zip) with draft declarations, and outputs/share.zip
  with aggregated results only (no subject-level data; safe to send to co-authors).

**Time:** seconds.

In [ ]:
# Setup: make the eegad package importable (also when it was not installed with `pip install -e .`)
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=RuntimeWarning)   # MNE notices about head size and 'boundary' events
warnings.filterwarnings("ignore", category=FutureWarning)

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "analysis_plan.json").exists()), None)
if _root is None:
    raise FileNotFoundError("Open this notebook from inside the repository (eeg-ad-demographics/notebooks).")
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

In [ ]:
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score

from eegad.config import load_context
from eegad.data import load_table
from eegad.reporting import fmt, fmt_ci, fmt_metrics, fmt_p, read_json, write_json

import datetime

import matplotlib.pyplot as plt

from eegad.config import library_versions
from eegad.figures import apply_style, confounding_figure, or_forest_figure, roc_figure
from eegad.reporting import save_figure
from eegad.submission import build_preprint_package, build_share_zip, declarations_draft, write_tripod

apply_style()
ctx = load_context("07")
log, out, plan = ctx.log, ctx.out, ctx.plan
loso = read_json(out / "results" / "loso.json")
inc = read_json(out / "results" / "incremental.json")
tr = read_json(out / "results" / "transport.json")
sens = read_json(out / "results" / "sensitivity.json")
TAB_CONF = pd.read_csv(out / "tables" / "T2b_confounding_by_design.csv", encoding="utf-8-sig")
TAB_AUC = pd.read_csv(out / "tables" / "auc_by_model.csv")
EEG_LABEL, COMB_LABEL = "EEG score (RF, LOSO)", "Demographics + EEG score"
CAL = plan["rq2"]["brainlat_matching"]["caliper_years"]
AGE_CAL = plan["rq2"]["greek_comparable_pairs"]["age_caliper_years"]

# Results written before their inputs were last updated come from an older run: re-run the notebook named
DEPENDENCIES = [("predictions/oof_BrainLat_RF_11.csv", ["derived/brainlat_epochs.csv"], "02"),
                ("results/incremental.json", ["predictions/oof_BrainLat_RF_11.csv", "predictions/oof_Greek_RF_11.csv"], "03"),
                ("tables/T2b_confounding_by_design.csv", ["predictions/oof_BrainLat_RF_11.csv",
                                                          "tables/confounding_design.json"], "04"),
                ("results/transport.json", ["derived/brainlat_epochs.csv", "derived/greek_epochs.csv"], "05"),
                ("results/sensitivity.json", ["results/incremental.json", "results/transport.json"], "06")]
for target, inputs, nb in DEPENDENCIES:
    newer = [i for i in inputs if (out / i).stat().st_mtime > (out / target).stat().st_mtime]
    if newer:
        log.alert(f"{target} is older than {newer}: re-run notebook {nb} (and the following ones) before the report.")

## Figures

In [ ]:
# Fig. 1 — ROC curves in both cohorts
panels = []
for name, demo_label, other, title in (("BrainLat", "Age + sex + education", "RQ3b", "BrainLat"),
                                       ("Greek", "Age + sex", "RQ3a", "Greek cohort (ds004504)")):
    b = load_table(out / "predictions" / f"baselines_{name}.csv")
    get = lambda m: (b[b.model == m]["y"].to_numpy(), b[b.model == m]["p"].to_numpy())
    tp = load_table(out / "predictions" / f"transport_{other}.csv")
    panels.append({"title": title, "eeg": get(EEG_LABEL), "other": (tp["y"].to_numpy(), tp["p"].to_numpy()),
                   "demo": (demo_label, *get(demo_label)), "combined": get(COMB_LABEL)})
fig = roc_figure(panels)
save_figure(fig, "fig1_roc_both_cohorts", out, log)
plt.show()

# Fig. 2 — OR per 1 SD of the EEG score beyond the demographics
I, P = inc["INC"], inc["R_POOL"]
ind = tr["INC_IND"]
rows = [("BrainLat (age, sex, education)", I["BrainLat"]["OR"], I["BrainLat"]["OR_lo"], I["BrainLat"]["OR_hi"]),
        ("Greek (age, sex)", I["Greek"]["OR"], I["Greek"]["OR_lo"], I["Greek"]["OR_hi"]),
        ("Pooled (cohort-specific demographics)", P["OR"], P["OR_lo"], P["OR_hi"]),
        ("BrainLat: score from the Greek model", *(ind["BrainLat (Greek model)"][k] for k in ("OR", "OR_lo", "OR_hi"))),
        ("Greek: score from the BrainLat model", *(ind["Greek (BrainLat model)"][k] for k in ("OR", "OR_lo", "OR_hi"))),
        ("BrainLat: + site", *(I["BrainLat + site"][k] for k in ("OR", "OR_lo", "OR_hi")))]
for r in sens[1:]:
    for cohort, label in (("BrainLat", "BrainLat"), ("Greek", "Greek")):
        rows.append((f"{label}: {r['variant']}", *r["or"][cohort]))
rows = [(lab, *(np.nan if v is None else v for v in vals)) for lab, *vals in rows]
fig = or_forest_figure(rows)
save_figure(fig, "fig2_incremental_or", out, log)
plt.show()

# Fig. 3 — AUC / concordance in the full cohort and in the confounding-free designs
SHORT = {"Full cohort": "Full cohort", f"Education-matched 1:1 (±{CAL} y)": "Matched pairs (education)",
         "Comparable-pair concordance (education ±2 y)": "Comparable pairs (education)",
         "Overlap-weighted (age, sex, education)": "Overlap weights (age, sex, educ.)",
         "Overlap-weighted (age, sex)": "Overlap weights (age, sex)",
         "Covariate-adjusted AUC (age, sex, education)": "Adjusted AUC (age, sex, educ.)",
         "Covariate-adjusted AUC (age, sex)": "Adjusted AUC (age, sex)",
         f"Comparable-pair concordance (same sex, age ±{AGE_CAL} y)": "Comparable pairs (sex, age)"}
fig = confounding_figure(TAB_CONF, SHORT)
save_figure(fig, "fig3_confounding", out, log)
plt.show()

## Table 2 and the results summary

In [ ]:
V = {k: v["reading"] for k, v in log.verdicts.items()}
B = inc["BOOT"]
T2 = []
for name, rq in (("BrainLat", "RQ1a"), ("Greek", "RQ1b")):
    r, bn = I[name], B[name]
    T2.append({"block": "RQ1 — incremental value", "cohort": name,
               "analysis": f"Within-cohort EEG score beyond {r['covariates']}",
               "estimate": f"OR/SD {fmt_ci(r['OR'], r['OR_lo'], r['OR_hi'], 2)}",
               "p": (f"null bootstrap {'one-sided ' if bn['one_sided'] else ''}{fmt_p(bn['p'])} "
                     f"({bn['replicates']} replicates); analytic {fmt_p(r['p_one_sided'] if bn['one_sided'] else r['p'])}"),
               "reading": V[rq]})
for key, cohort in (("BrainLat (Greek model)", "BrainLat"), ("Greek (BrainLat model)", "Greek")):
    r = ind[key]
    T2.append({"block": "RQ1 — incremental value", "cohort": cohort,
               "analysis": f"Score from the other cohort's model beyond {r['covariates']} (independent)",
               "estimate": f"OR/SD {fmt_ci(r['OR'], r['OR_lo'], r['OR_hi'], 2)}",
               "p": fmt_p(r["p_one_sided"]) + " one-sided" if cohort == "Greek" else fmt_p(r["p"]),
               "reading": "independent confirmation"})
rs = I["BrainLat + site"]
T2.append({"block": "RQ1 — incremental value", "cohort": "BrainLat", "analysis": "Check: + site",
           "estimate": f"OR/SD {fmt_ci(rs['OR'], rs['OR_lo'], rs['OR_hi'], 2)}", "p": fmt_p(rs["p"]), "reading": "check"})
T2.append({"block": "RQ1 — incremental value", "cohort": "Pooled",
           "analysis": "EEG score beyond cohort-specific demographics (common EEG effect)",
           "estimate": f"OR/SD {fmt_ci(P['OR'], P['OR_lo'], P['OR_hi'], 2)}",
           "p": f"{fmt_p(P['p'])}; cohort × EEG {fmt_p(inc['R_HET']['p'])}", "reading": V["RQ1c"]})
for _, l in TAB_AUC.iterrows():
    T2.append({"block": "Discrimination (logistic: leave-one-out; EEG: LOSO)", "cohort": l["cohort"],
               "analysis": l["model"], "estimate": f"AUC {fmt_ci(l['AUC'], l['lo'], l['hi'])}", "p": "", "reading": ""})
for name, res in loso.items():
    T2.append({"block": "Discrimination (logistic: leave-one-out; EEG: LOSO)", "cohort": name,
               "analysis": f"EEG score, threshold {ctx.threshold}", "estimate": fmt_metrics(res["metrics"]), "p": "",
               "reading": ""})
for _, l in TAB_CONF.iterrows():
    T2.append({"block": "RQ2 — confounding removed by design", "cohort": l["cohort"], "analysis": l["analysis"],
               "estimate": f"{fmt_ci(l['AUC'], l['lo'], l['hi'])} ({l['n']})", "p": "",
               "reading": "" if l["analysis"] == "Full cohort" else V["RQ2a" if l["cohort"] == "BrainLat" else "RQ2b"]})
for k, t in tr["TRANSP"].items():
    T2.append({"block": "RQ3 — transportability", "cohort": "Greek" if k == "RQ3a" else "BrainLat",
               "analysis": t["label"], "estimate": f"AUC {fmt_ci(*t['AUC'])}; {fmt_metrics(t['metrics'])}",
               "p": f"calibration slope {t['calibration_slope']:.2f}; calibration-in-the-large {t['citl']:.2f}",
               "reading": V[k]})
for k, t in tr["TRANSP_STD"].items():
    T2.append({"block": "RQ3 — transportability", "cohort": "Greek" if k == "RQ3a" else "BrainLat",
               "analysis": t["label"], "estimate": f"AUC {fmt_ci(*t['AUC'])}",
               "p": f"calibration slope {t['calibration_slope']:.2f}; calibration-in-the-large {t['citl']:.2f}",
               "reading": "secondary"})
TAB2 = pd.DataFrame(T2)
TAB2.to_csv(out / "tables" / "T2_main_results.csv", index=False, encoding="utf-8-sig")
display(TAB2)

# Summary
run_log = ctx.read_run_log()
frozen = read_json(out / "plan_frozen.json")
dl = read_json(out / "ds004504_download.json") if (out / "ds004504_download.json").exists() else {}
versions = library_versions()
esc = lambda s: str(s).replace("|", "\\|")
lines = [f"# Results — EEG and demographics in two cohorts — {datetime.datetime.now().isoformat(timespec='seconds')}", "",
         f"**{plan['working_title']}**", "",
         f"Mode: {'QUICK TEST (DO NOT USE)' if ctx.quick else 'full'} | analysis plan {frozen['hash']}, frozen on "
         f"{frozen['frozen_at']} | Greek labels first read on {run_log.get('greek_labels_first_read', 'n/a')}", "",
         f"Original analysis (provenance): plan {plan['provenance']['original_hash']}, frozen on "
         f"{plan['provenance']['original_frozen_at']}", "",
         f"ds004504 v{dl.get('version', 'n/a')} (doi:{dl.get('expected_doi', 'n/a')}); DOI in folder: "
         f"{dl.get('doi_in_folder') or 'n/a'}; {dl.get('files_ok')} of {dl.get('files_in_manifest')} files verified by SHA-256", "",
         " | ".join(f"{k} {v}" for k, v in versions.items() if k != "system"), "",
         "## Readings by the plan's rules", "", "| Question | Reading |", "|---|---|"]
order = ["RQ1a", "RQ1b", "RQ1c", "RQ1", "RQ2a", "RQ2b", "RQ3a", "RQ3b"]
lines += [f"| {k} | {V[k]} |" for k in order if k in V]
lines += ["", "## Results", "", "| Notebook | Section | Item | Value | Where in the paper | Note |", "|---|---|---|---|---|---|"]
entries = sorted(log.entries, key=lambda e: e["notebook"])
lines += ["| " + " | ".join(esc(e[c]) for c in ("notebook", "section", "item", "value", "where", "note")) + " |"
          for e in entries]
lines += ["", "## Alerts", ""] + ([f"- ({a['notebook']}) {a['message']}" for a in log.alerts] or ["- none"])
if plan["change_history"] or len(run_log.get("freezes", [])) > 1:
    lines += ["", "## Changes to the analysis plan after the first freeze", ""]
    lines += [f"- freeze: {c['frozen_at']} (hash {c['hash']})" for c in run_log.get("freezes", [])]
    lines += [f"- {h}" for h in plan["change_history"]]
(out / "results_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("Readings:", V)
print("Summary:", out / "results_summary.md")

## TRIPOD+AI checklist, preprint package and shareable zip

In [ ]:
tripod = write_tripod(out)
print(tripod["status"].value_counts().to_string())
original = {"hash": plan["provenance"]["original_hash"], "frozen_at": plan["provenance"]["original_frozen_at"]}
decl = declarations_draft(frozen["hash"], frozen["frozen_at"], run_log.get("greek_labels_first_read", "n/a"), original)
pkg, f_pkg = build_preprint_package(out, decl)
f_share = build_share_zip(out)
print(f"Preprint package: {pkg} ({f_pkg.name})")
print(f"Aggregated results to share: {f_share}")
print("To do by hand: manuscript PDF, the [FILL IN] fields of the declarations, the Zenodo DOI and the medRxiv deposit.")